# LEDNet: Lightweight Edge Detection Network

**Bio-Inspired Deep Learning Model**

**Bio-Inspiration**: First and second visual pathways (magno/parvo)  
**Deep Learning Enhancement**: Compact receptive field enhanced network  
**Improvement Area**: Improved edge localization and reduced computational costs

Inspired by magnocellular (motion) and parvocellular (detail) pathways.

In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR = Path('..') / 'bio DL' / 'outputs' / 'LEDNet'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## LEDNet: Magno (M) and Parvo (P) Pathways

In [ ]:
class LEDNet(nn.Module):
    """Magno (M) and Parvo (P) pathways for edge detection"""
    def __init__(self):
        super().__init__()
        # Magnocellular pathway (motion, fast, low-res)
        self.magno_conv1 = nn.Conv2d(3, 32, 5, padding=2)
        self.magno_conv2 = nn.Conv2d(32, 64, 5, padding=2)
        # Parvocellular pathway (color, detail, high-res)
        self.parvo_conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.parvo_conv2 = nn.Conv2d(32, 64, 3, padding=1)
        # Integration
        self.fusion = nn.Conv2d(128, 64, 1)
        self.edge = nn.Conv2d(64, 1, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        # M pathway
        m = F.relu(self.magno_conv1(x))
        m = F.relu(self.magno_conv2(m))
        # P pathway
        p = F.relu(self.parvo_conv1(x))
        p = F.relu(self.parvo_conv2(p))
        # Combine
        combined = torch.cat([m, p], dim=1)
        fused = F.relu(self.fusion(combined))
        return torch.sigmoid(self.edge(fused))

model = LEDNet().to(DEVICE).eval()
params = sum(p.numel() for p in model.parameters())
print(f"✓ LEDNet: {params:,} params (lightweight)")

In [ ]:
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir, self.gt_dir = root / split / 'images', root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))[:20]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt = cv2.imread(str(self.gt_dir / img_path.name.replace('.jpg', '.png')), 0)
        gt = gt.astype(np.float32) / 255.0 if gt is not None else np.zeros(img.shape[:2], dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

loader = DataLoader(EdgeDataset(Path('..') / 'datasets' / 'HED_Small', 'test'), batch_size=1)
preds, gts = [], []
with torch.no_grad():
    for imgs, gt, _ in tqdm(loader):
        preds.extend([model(imgs.to(DEVICE))[i,0].cpu().numpy() for i in range(imgs.shape[0])])
        gts.extend([gt[i].cpu().numpy() for i in range(gt.shape[0])])

def eval_m(preds, labels):
    t, ois, ap, al = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l = cv2.dilate((l>0.5).astype(np.float32), np.ones((3,3))).flatten()
        p = cv2.GaussianBlur(p, (3,3), 0).flatten()
        ap.append(p); al.append(l)
        ois.append(max([2*np.sum((p>=th)*l)/(2*np.sum((p>=th)*l)+np.sum((p>=th)*(1-l))+np.sum((p<th)*l)+1e-8) for th in t]))
    fp, fl = np.concatenate(ap), np.concatenate(al)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': ods[0], 'ODS_thresh': ods[1], 'OIS': np.mean(ois), 'AP': average_precision_score(fl, fp) if np.sum(fl)>0 else 0}

m = eval_m(preds, gts)
print(f"\nLEDNet: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f} | Params={params:,}")

import json
with open(OUTPUT_DIR / 'lednet_metrics.json', 'w') as f:
    json.dump({'model': 'LEDNet', 'bio': 'Magno/Parvo pathways', 'improvement': 'Edge localization + efficiency', 'metrics': m, 'params': params}, f, indent=2)
print("✅ LEDNet complete!")